In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader

from diffusers import UNet2DModel

sys.path.append("../")

from oxels.analytical_benchmark import TwoShapes
from oxels.datasets import DGPerspectiveDataset
from oxels.losses import original_loss
from oxels.visualization.plot_channels import plot_channels

In [ ]:
dataset = DGPerspectiveDataset("data", "ColoredMNIST", 32, 32, split="train", domain_split="id", seed=0)

In [ ]:
# view1, view2, indices, flags, mask1, mask2 = dataset[0]
# print(view1.shape, view2.shape, indices.shape, flags.shape, mask1.shape, mask2.shape)
# plt.subplot(221)
# plt.imshow(view1)
# plt.axis("off")
# plt.subplot(222)
# plt.imshow(view2)
# plt.axis("off")
# plt.subplot(223)
# plt.imshow((view2.reshape((-1,3))[indices]*flags[:,None]).reshape(view2.shape))
# plt.axis("off")
# plt.subplot(224)
# plt.imshow(mask1.reshape(view1.shape[:2]))
# plt.axis("off")
# plt.show()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
batch_size = 8

loader = DataLoader(
    dataset,
    batch_size=batch_size,
    num_workers=len(os.sched_getaffinity(0)),
    pin_memory=True,
    shuffle=True,
    drop_last=True
)

In [ ]:
# import wandb

# # Start a new wandb run to track this script.
# run = wandb.init(
#     entity="kl_divergence-rensselaer-polytechnic-institute",
#     project="oxels",
# )

In [ ]:
network = UNet2DModel(
    sample_size=(32, 32),
    in_channels=3,
    out_channels=8,
    layers_per_block=2,
    norm_num_groups=4,
    block_out_channels=(16, 32, 64),
    time_embedding_dim=1,
    down_block_types=(
        "DownBlock2D", "DownBlock2D", "DownBlock2D"
    ),
    up_block_types=(
        "UpBlock2D", "UpBlock2D", "UpBlock2D",
    ),
).to(device)

In [ ]:
dummy_t = torch.tensor([0]).to(device)

In [ ]:
view1, view2, permutation, flags, mask1, mask2 = next(iter(loader))

In [ ]:
%%time
with torch.no_grad():
    oxels1 = network(view1.to(device), dummy_t)["sample"]
    oxels2 = network(view2.to(device), dummy_t)["sample"]
    print(original_loss(oxels1, oxels2, permutation.to(device), flags.to(device), mask1.to(device), mask2.to(device)))

In [ ]:
EPOCHS = 100
SAVE_EVERY = 25
LEARNING_RATE = 9e-5
LOG_EVERY = 5

optimizer = torch.optim.Adam(network.parameters(), lr=LEARNING_RATE)


scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

for epoch in range(EPOCHS):
    network.train()
    total_loss = 0.0

    for step, (view1, view2, permutation, flags, mask1, mask2) in enumerate(loader):
        view1 = view1.to(device)
        view2 = view2.to(device)
        permutation = permutation.to(device)
        flags = flags.to(device)
        mask1 = mask1.to(device)
        mask2 = mask2.to(device)

        optimizer.zero_grad()
        oxels1 = network(view1, dummy_t)["sample"]
        oxels2 = network(view2, dummy_t)["sample"]

        loss = original_loss(oxels1, oxels2, permutation, flags, mask1, mask2)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # if step % LOG_EVERY == 0:
        #     run.log(data={"loss": loss})

    scheduler.step()

    avg_loss = total_loss / len(loader)

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {avg_loss:.3f}, LR: {scheduler.get_last_lr()[0]:.2e}")

    # Save model checkpoint
    if epoch % SAVE_EVERY == 0:
        torch.save(network.state_dict(), f"../checkpoints/checkpoint_epoch_{epoch+1}.pth")

In [ ]:
f = plot_channels(oxels2[0])

In [ ]:
ood_dataset = DGPerspectiveDataset("data", "ColoredMNIST", 32, 32, domain_split="ood", seed=0)

In [ ]:
view, *_ = ood_dataset[0]
view = view[None]
view = torch.as_tensor(view, device=device)
view = network(view, dummy_t)
view = view.sample
view.shape

In [ ]:
f = plot_channels(view[0])